<a href="https://www.kaggle.com/code/cartelsmith/adaboost-price-prediction-r2-91-7?scriptVersionId=336582003" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "modern_laptops_kaggle_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sohaibdevv/modern-laptops-hardware-and-pricing-dataset-2026",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='white',context='notebook' )

import pandas as pd
#Prevent wrapping dataframes across multiple lines
pd.set_option('display.max_columns',None)
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

/tmp/ipykernel_58/641165501.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [2]:
print(df.info())
print(f'\n\nNumber of unique values by column:\n{df.nunique()}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Laptop_ID          200 non-null    int64 
 1   Brand              200 non-null    object
 2   Model              200 non-null    object
 3   CPU                200 non-null    object
 4   RAM                200 non-null    object
 5   Storage            200 non-null    object
 6   GPU                200 non-null    object
 7   Price_USD          200 non-null    int64 
 8   Performance_Level  200 non-null    object
dtypes: int64(2), object(7)
memory usage: 14.2+ KB
None


Number of unique values by column:
Laptop_ID            200
Brand                 11
Model                 31
CPU                   24
RAM                    6
Storage                7
GPU                   20
Price_USD            187
Performance_Level      4
dtype: int64


In [3]:
category_columns = ['RAM','Storage','Performance_Level']
df[category_columns] = df[category_columns].astype('category')
print('---'*20)
df.info()

------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Laptop_ID          200 non-null    int64   
 1   Brand              200 non-null    object  
 2   Model              200 non-null    object  
 3   CPU                200 non-null    object  
 4   RAM                200 non-null    category
 5   Storage            200 non-null    category
 6   GPU                200 non-null    object  
 7   Price_USD          200 non-null    int64   
 8   Performance_Level  200 non-null    category
dtypes: category(3), int64(2), object(4)
memory usage: 10.9+ KB


In [4]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error,r2_score,mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import OneHotEncoder

In [5]:
#Prepping Columns for Model (Preprocessing)
str_cols = df.select_dtypes(include=['object','string']).columns.to_list()
str_cols = [col for col in str_cols if col not in category_columns]

#Building Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), category_columns),
        ('str', OneHotEncoder(handle_unknown='ignore', sparse_output=False), str_cols)
    ],
    remainder='passthrough')

In [6]:
#Splitting Data for Model
df = df.drop(columns='Laptop_ID')
x = df.drop(columns='Price_USD')
y = df['Price_USD']

print(f'This is the shape of the full feature set pre-split:\n{x.shape}')
print(f'This is the shape of the full target set pre-split:\n{y.shape}')
print('***'*25)
#Actual Split
X_train,X_test, y_train,y_test = train_test_split(x,y,random_state=12,train_size=.8)

print(f'\n\nThis is the shape of the training feature set post-split:\n{X_train.shape}')
print(f'This is the shape of the training target set post-split:\n{y_train.shape}')
print('***'*25)

This is the shape of the full feature set pre-split:
(200, 7)
This is the shape of the full target set pre-split:
(200,)
***************************************************************************


This is the shape of the training feature set post-split:
(160, 7)
This is the shape of the training target set post-split:
(160,)
***************************************************************************


In [7]:
#Defining the Model
model = AdaBoostRegressor(learning_rate=.8, random_state=12) #Use 'help(AdaBoostRegressor)' to look at estimator params


#Setting up Pipeline
pipe = make_pipeline(preprocessor, model)

#Fitting Pipeline
pipe.fit(X_train,y_train)

#Predicting with Model
price_predict = pipe.predict(X_test)

In [8]:
#Evaluating the Model
metrics = [
    mean_absolute_percentage_error, 
    mean_absolute_error, 
    r2_score, 
    mean_squared_error
]
print("Eval Metrics:")
for metric_func in metrics:
    score = metric_func(y_true=y_test, y_pred=price_predict)
    print(f"{metric_func.__name__}: {score}")

Eval Metrics:
mean_absolute_percentage_error: 0.23834318625057946
mean_absolute_error: 236.40353797629405
r2_score: 0.8905652010134757
mean_squared_error: 100665.13131936945


In [9]:
#Plotting Permutation Importance
from sklearn.inspection import permutation_importance

# 1. Perform permutation importance
#We use 'pipe' (the full pipeline) and 'X_test'/'y_test'
result = permutation_importance(
    pipe, 
    X_test, 
    y_test, 
    n_repeats=10, 
    random_state=42, 
    n_jobs=-1  # Uses all CPU cores for faster computation
)

# 2. Prepare data for plotting
#We use X_test.columns to ensure we have the original feature names
df_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
})

# Sort by mean importance
df_importance = df_importance.sort_values(by='importance_mean', ascending=True)

# 3. Create the Plotly figure
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_importance['importance_mean'],
    y=df_importance['feature'],
    orientation='h',
    error_x=dict(type='data', array=df_importance['importance_std']),
    marker_color='royalblue'
))

fig.update_layout(
    title='Permutation Importance (via Pipeline)',
    xaxis_title='Decrease in Model Score',
    yaxis_title='Features',
    template='plotly_white',
    height=600
)

fig.show(renderer='iframe_connected')